In [2]:
!pip uninstall NN_UTCI -y

Found existing installation: NN_UTCI 0.1.0
Uninstalling NN_UTCI-0.1.0:
  Successfully uninstalled NN_UTCI-0.1.0


In [3]:
!pip install git+https://github.com/bikempastine/nn_utci.git

  Cloning https://github.com/bikempastine/nn_utci.git to c:\users\nerc-user\appdata\local\temp\pip-req-build-gp8kwv69
  Resolved https://github.com/bikempastine/nn_utci.git to commit ea2256b25418bae19dd6b603bdbdb47be61a9dca
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for NN_UTCI: filename=nn_utci-0.1.0-py3-none-any.whl size=44575 sha256=2a6a2fbc009386a06cbe75cde2d5f74b90213514d95d4dd7b8139318fd341dc4
  Stored in directory: C:\Users\nerc-user\AppData\Local\Temp\pip-ephem-wheel-cache-59ih1d65\wheels\45\b1\ed\0e2170742686baddfabdbb69c1e84c98b9e45c030915bb75bf
Successfully built NN_UTCI


  Running command git clone --filter=blob:none --quiet https://github.com/bikempastine/nn_utci.git 'C:\Users\nerc-user\AppData\Local\Temp\pip-req-build-gp8kwv69'


In [1]:
# Scientific stack
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# External models
from NN_UTCI import NN_UTCI


FileNotFoundError: [Errno 2] No such file or directory: 'assets/utci_nn_weights.pth'

## Test on the data from Brode et al

In [2]:
esm4 = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\ESM_4_Table_Offset.DAT', sep='\t', comment='*')
test = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\UTCI-Test-Data.txt', sep='\t', comment='#')

In [3]:
test['Tr'] = test['Tr-Ta'] + test['Ta']  # convert to absolute MRT

### Run model on the test data to check the output is as expected

In [4]:
test["Offset_NN"] = NN_UTCI(Ta=test['Ta'], Tr=test['Tr'], va=test['va'], rH=test['rH'], oob='nan')

In [5]:
test_analysis = test.copy()
y_true = test_analysis['UTCI']
y_pred = test_analysis['UTCI_polynomial']
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse:.3f} °C")

RMSE: 2.777 °C


### Test Clamping

In [9]:
NN_UTCI(Ta=70, Tr=28, va=10, rH=100, oob='clamp')

np.float64(80.78778839111328)

In [10]:
NN_UTCI(Ta=70, Tr=28, va=10, rH=110, oob='clamp')

np.float64(80.78778839111328)

### Input shape mismatch - should return an error

In [11]:
NN_UTCI(Ta=np.array([20.0, 25.0]),
        Tr=np.array([25.0]),          # wrong length
        va=np.array([ 1.0,  2.0]),
        rH=np.array([50.0, 60.0]))

ValueError: All inputs must have the same length, got shapes: {1, 2}

### Single input as an array

In [12]:
NN_UTCI(Ta=np.array([20.0]),
            Tr=np.array([25.0]),
            va=np.array([ 1.0]),
            rH=np.array([50.0]))

array([21.4264946])

### One valid row

#### Nan

In [13]:
NN_UTCI(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="nan")

array([21.4264946,        nan])

#### clamping

In [14]:
NN_UTCI(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="clamp")

array([21.4264946 , 76.80784607])

## Edge cases, One valid, one out of bounds output

In [16]:
NN_UTCI(
            Ta=np.array([-50.0, 50.0]),
            Tr=np.array([0.0, 135.0]),
            va=np.array([  1.0,   35.0]),
            rH=np.array([ 50.0,  50.0]),
            oob="nan",
        )

array([-42.43745804,          nan])